# 04 · Observabilidade: custo, latência e volume por tool

**Objetivo:** responder, com dados do LangSmith, *quanto custa e quanto demora cada etapa* do agente —
e deixar evidente o contraste **determinística × RAG × LLM**.

### O que o LangSmith já dá pronto × o que precisou de camada extra

| Necessidade | Nativo no LangSmith? | Como |
|---|---|---|
| Árvore do trace, inputs/outputs de cada tool | ✅ | aba do projeto → clicar no trace |
| Latência de cada run (tool, nó, LLM) | ✅ | em cada run; gráficos p50/p99 na aba **Monitor** |
| Tokens e custo de cada chamada de LLM / custo total do trace | ✅ | calculado pela tabela de preços do LangSmith |
| Custo do **embedding** do RAG | ⚠️ parcial | só aparece porque registramos `usage_metadata` no run (ver `receitas/rag.py`) |
| Volume de chamadas por tool, filtros por tag/metadata | ✅ | Monitor / Dashboards customizados agrupando por `name` |
| Comparar experimentos (modelo/prompt) | ✅ | Datasets & Experiments → Compare |
| **Custo médio por tool incluindo os LLMs aninhados** (ex.: o LLM dentro de `criar_receita`) | ❌ | camada extra: `receitas/custos.py` (este notebook) |
| Tabela lado a lado determinística × RAG × LLM, KPIs próprios, export para BI | ❌ | este notebook + dashboard Streamlit (`dashboard/app.py`) |

Regra prática: **use a UI do LangSmith para investigar um trace e acompanhar a operação**; exporte os dados
(`Client.list_runs`) só para as agregações que ela não faz.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # adiciona a raiz do projeto ao path

from receitas import config

## 1. Gerar tráfego "de produção" (simulado)

Aqui a geladeira **muda a cada pedido** (sem seed) — é o mundo real, dinâmico. Avaliação comparativa usa o dataset
congelado (notebooks 03/05); monitoramento usa tráfego real.

Se o notebook 05 já gerou uma política de roteamento, ela é usada; senão, usamos uma política inicial "intuitiva".

In [5]:
import json, random
from receitas.agente import criar_alvo
from receitas.dataset import CENARIOS
from receitas.geladeira import gerar_geladeira

if config.ARQ_POLITICA.exists():
    politica = json.loads(config.ARQ_POLITICA.read_text())["politica"]
    print("Política do notebook 05:", politica)
else:
    politica = {"facil": config.MODELO_SIMPLES, "medio": config.MODELO_INTERMEDIARIO, "dificil": config.MODELO_FRONTEIRA}
    print("Política inicial (ainda não validada):", politica)

N_PEDIDOS = 12
alvo = criar_alvo(politica=politica)
sorteio = random.Random()  # sem seed: pedidos diferentes a cada execução

saidas = []
for i in range(N_PEDIDOS):
    cenario = sorteio.choice(CENARIOS)
    geladeira = gerar_geladeira(n_itens=sorteio.randint(*cenario["n_itens"]))
    saida = alvo({"pergunta": cenario["pergunta"], "geladeira": geladeira})
    saidas.append({"pedido": cenario["pergunta"][:60], "roteado": saida["complexidade_roteada"],
                   "modelo": saida["modelo"], "tools": saida["tools_chamadas"],
                   "custo_usd": saida["custo_usd"], "latencia_s": saida["latencia_s"]})
    print(f"{i+1:2d}/{N_PEDIDOS} {saida['modelo']:13s} US$ {saida['custo_usd']:.5f} {saida['latencia_s']:5.1f}s {saida['tools_chamadas']}")

Política inicial (ainda não validada): {'facil': 'gpt-4.1-nano', 'medio': 'gpt-4.1-mini', 'dificil': 'gpt-4.1'}
 1/12 gpt-4.1-nano  US$ 0.00076   7.3s ['verificar_validade', 'check_geladeira', 'criar_receita']
 2/12 gpt-4.1-mini  US$ 0.00555  10.5s ['verificar_validade', 'check_geladeira', 'buscar_receitas_similares', 'criar_receita']
 3/12 gpt-4.1-nano  US$ 0.00069   6.5s ['check_geladeira', 'verificar_validade', 'criar_receita']
 4/12 gpt-4.1-nano  US$ 0.00068   6.7s ['check_geladeira', 'verificar_validade', 'criar_receita']
 5/12 gpt-4.1-nano  US$ 0.00025   2.4s ['verificar_validade']
 6/12 gpt-4.1-mini  US$ 0.00514  39.1s ['verificar_validade', 'check_geladeira', 'check_geladeira', 'criar_receita']
 7/12 gpt-4.1-mini  US$ 0.00303   8.2s ['verificar_validade', 'check_geladeira', 'criar_receita']
 8/12 gpt-4.1-nano  US$ 0.00081   7.0s ['check_geladeira', 'verificar_validade', 'criar_receita', 'criar_receita']
 9/12 gpt-4.1-nano  US$ 0.00072   7.5s ['check_geladeira', 'verificar_valid

## 2. Exportar os runs do LangSmith
Os traces são enviados em segundo plano; esperamos o envio terminar e damos alguns segundos para a ingestão.

In [9]:
import time
import pandas as pd
from langchain_core.tracers.langchain import wait_for_all_tracers
from receitas.custos import exportar_runs, resumo_por_etapa, resumo_por_trace, salvar_export

wait_for_all_tracers()
time.sleep(20)

runs = exportar_runs(horas=6)  # traces das últimas 6 horas
print(f"{len(runs)} runs exportados | {runs['trace_id'].nunique()} traces")
runs[["nome", "run_type", "etapa", "modelo", "latencia_s", "tokens_entrada", "tokens_saida", "custo_usd"]].head(12)

### Como o custo é atribuído a cada tool

Cada run tem um `dotted_order` — o caminho completo da raiz do trace até ele. Para cada chamada de LLM ou embedding,
subimos esse caminho até achar a tool (ou o roteador) "dona" dela. LLM sem dona = **o agente decidindo o próximo passo**.
O custo é recalculado com a nossa tabela de preços (`config.PRECOS_POR_MILHAO`) e conferido contra o do LangSmith abaixo.

In [ ]:
resumo = resumo_por_etapa(runs)
resumo.style.format({
    "latencia_p50_s": "{:.4f}", "latencia_p95_s": "{:.4f}", "tokens": "{:,.0f}",
    "custo_total_usd": "US$ {:.5f}", "custo_por_chamada_usd": "US$ {:.6f}",
})

In [ ]:
# Compara o custo calculado aqui com o do LangSmith
llms = runs[runs["run_type"] == "llm"]
print(f"Nosso cálculo:   US$ {llms['custo_usd'].sum():.5f}")
print(f"LangSmith:       US$ {llms['custo_langsmith_usd'].sum():.5f}")
print("(diferenças pequenas vêm de tokens em cache ou de preços diferentes na tabela de cada um)")

## 3. O contraste, em gráficos
Latência em **escala logarítmica**: uma tool determinística (~ms) e uma de LLM (~s) diferem em mais de 1000× — em escala linear, a barra das determinísticas seria invisível.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from receitas.graficos import aplicar_estilo, barras_horizontais, COR_POR_TIPO

aplicar_estilo()
dados = resumo.sort_values("latencia_p50_s")
cores = [COR_POR_TIPO.get(t, "#999999") for t in dados["tipo"]]

fig, eixos = plt.subplots(1, 3, figsize=(14, 3.6), sharey=False)
barras_horizontais(eixos[0], dados.index, dados["latencia_p50_s"], cores, "{:.3g} s", "Latência p50 por chamada (log)", log=True)
barras_horizontais(eixos[1], dados.index, dados["custo_por_chamada_usd"], cores, "US$ {:.2g}", "Custo médio por chamada")
barras_horizontais(eixos[2], dados.index, dados["chamadas"], cores, "{:.0f}", "Volume de chamadas")
for ax in eixos[1:]:
    ax.set_yticklabels([])
fig.legend(handles=[Patch(color=c, label=t) for t, c in COR_POR_TIPO.items() if t in set(dados["tipo"])],
           loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.show()

In [ ]:
# Participação de cada etapa no custo total
participacao = (resumo["custo_total_usd"] / resumo["custo_total_usd"].sum() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 2.8))
barras_horizontais(ax, participacao.index, participacao.values,
                   [COR_POR_TIPO.get(resumo.loc[e, "tipo"]) for e in participacao.index], "{:.1f}%", "Participação no custo total")
plt.tight_layout()
plt.show()